# 🏢 Business Entity Resolution — ML Challenge Pipeline

Each pipeline stage runs in its **own cell**. If a cell fails or the session disconnects, all previous state is preserved in memory. Just re-run the failed cell.

**Dataset paths:**
- Train: `/kaggle/input/datasets/cdbharath234/dataset3/student_resource/dataset/train/`
- Test: `/kaggle/input/datasets/cdbharath234/dataset3/student_resource/dataset/test/`

---
## 📦 Cell 1: Clone Repository & Install Dependencies

In [ ]:
import os

# Navigate to Kaggle working directory
%cd /kaggle/working

# Clean old clone if re-running
!rm -rf ml_challenge

# Clone GitHub repository
!git clone https://github.com/BharathPESU/ml_challenge.git

# Navigate to source code directory
%cd /kaggle/working/ml_challenge/Error_404_Not_Found_submission/code/business_entity_resolution

# Switch to your branch and pull latest code
!git checkout Bharath.2004
!git pull origin Bharath.2004

# Install dependencies
%pip install -r requirements.txt --quiet

print("\n✓ Repository cloned and dependencies installed!")
print("✓ Current directory:", os.getcwd())

---
## ⚙️ Cell 2: Initialize Config & Pipeline State

This cell sets up the configuration and creates the shared `state` object that persists across all subsequent cells.

In [ ]:
import sys, os, logging
from IPython.display import display

# Dynamically resolve src path for both local execution and Kaggle
possible_src_paths = [
    os.path.abspath("Error_404_Not_Found_submission/code/business_entity_resolution/src"),
    "/kaggle/working/ml_challenge/Error_404_Not_Found_submission/code/business_entity_resolution/src",
    os.path.abspath("src"),
]

for p in possible_src_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from config import Config
from pipeline import PipelineState

# =====================================================
#  CONFIGURATION — Edit these values as needed
# =====================================================

# Dataset paths on Kaggle
BASE_DIR = "/kaggle/input/datasets/cdbharath234/dataset3/student_resource/dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")
OUTPUT_DIR = "/kaggle/working/output"

# Sample size: set to 100 for sanity check, set to None for full dataset run
SAMPLE_SIZE = 100  # Change to None for full pipeline

# =====================================================

# Create config
cfg = Config()
cfg.data.train_source1_path = os.path.join(TRAIN_DIR, "train_source1.tsv")
cfg.data.train_source2_path = os.path.join(TRAIN_DIR, "train_source2.tsv")
cfg.data.train_source3_path = os.path.join(TRAIN_DIR, "train_source3.tsv")
cfg.data.train_ground_truth_path = os.path.join(TRAIN_DIR, "train_ground_truth.tsv")
cfg.data.test_source1_path = os.path.join(TEST_DIR, "test_source1.tsv")
cfg.data.test_source2_path = os.path.join(TEST_DIR, "test_source2.tsv")
cfg.data.test_source3_path = os.path.join(TEST_DIR, "test_source3.tsv")
cfg.data.output_dir = OUTPUT_DIR
cfg.runtime.debug_sample_size = SAMPLE_SIZE

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Configure logging
cfg.configure_logging()

# Create shared pipeline state (persists across all cells)
state = PipelineState()

print("✓ Config initialized")
print(f"  Train dir : {TRAIN_DIR}")
print(f"  Test dir  : {TEST_DIR}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Sample size: {SAMPLE_SIZE or 'FULL DATASET'}")

---
## 📥 Cell 3 — Stage 1: Load Training Data

In [ ]:
from pipeline import load_training_data

print("⏳ Stage 1: Loading training data...")
load_training_data(cfg, state)
print(f"✅ Stage 1 Complete — S1: {len(state.train_s1)} | S2: {len(state.train_s2)} | S3: {len(state.train_s3)} | GT: {len(state.gt)}")

---
## 🧹 Cell 4 — Stage 2: Clean Training Data

In [ ]:
from pipeline import clean_training_data

print("⏳ Stage 2: Cleaning training data...")
clean_training_data(cfg, state)
print(f"✅ Stage 2 Complete — S1_clean: {len(state.train_s1_clean)} | S2_clean: {len(state.train_s2_clean)} | S3_clean: {len(state.train_s3_clean)}")

---
## ✂️ Cell 5 — Stage 3: Train/Validation Split

In [ ]:
from pipeline import split_training_validation

print("⏳ Stage 3: Splitting into train/validation...")
split_training_validation(cfg, state)
print(f"✅ Stage 3 Complete — Train S1 IDs: {len(state.train_s1_ids)} | Val S1 IDs: {len(state.val_s1_ids)}")

---
## 🔍 Cell 6 — Stage 4: Generate Candidate Pairs (Blocking)

In [ ]:
from pipeline import generate_candidates

print("⏳ Stage 4: Generating candidate pairs via TF-IDF blocking...")
generate_candidates(cfg, state)
print(f"✅ Stage 4 Complete — Train candidates: {len(state.train_candidates)} | Val candidates: {len(state.val_candidates)}")

---
## 📊 Cell 7 — Stage 5: Evaluate Candidate Recall

In [ ]:
from pipeline import evaluate_candidate_recall

print("⏳ Stage 5: Evaluating candidate recall...")
evaluate_candidate_recall(cfg, state)
print("✅ Stage 5 Complete — Check recall numbers in logs above.")

---
## 🏗️ Cell 8 — Stage 6: Build Training Pairs

In [ ]:
from pipeline import build_training_pairs

print("⏳ Stage 6: Building training and validation pairs...")
build_training_pairs(cfg, state)
print(f"✅ Stage 6 Complete — Train pairs: {len(state.train_pairs)} | Val pairs: {len(state.val_pairs)}")

---
## 🧮 Cell 9 — Stage 7: Extract Features

In [ ]:
from pipeline import build_features

print("⏳ Stage 7: Extracting pairwise features...")
build_features(cfg, state)
print(f"✅ Stage 7 Complete — Feature columns ({len(state.feature_columns)}): {state.feature_columns[:5]}...")

---
## 🤖 Cell 10 — Stage 8: Train XGBoost Model

In [ ]:
from pipeline import train_xgboost

print("⏳ Stage 8: Training XGBoost classifier...")
train_xgboost(cfg, state)
print(f"✅ Stage 8 Complete — Model saved to: {state.model_path}")

---
## 📈 Cell 11 — Stage 9: Score Validation Set

In [ ]:
from pipeline import score_validation

print("⏳ Stage 9: Scoring validation candidates...")
score_validation(cfg, state)
print(f"✅ Stage 9 Complete — Probabilities generated for {len(state.val_probabilities)} candidates")

---
## 🎯 Cell 12 — Stage 10: Optimize Threshold

In [ ]:
from pipeline import optimize_threshold

print("⏳ Stage 10: Sweeping thresholds for best F0.5...")
optimize_threshold(cfg, state)
print(f"✅ Stage 10 Complete — Best threshold: {state.best_threshold:.4f}")

---
## 🔎 Cell 13 — Stage 11: Error Analysis

In [ ]:
from pipeline import perform_error_analysis

print("⏳ Stage 11: Performing error analysis...")
perform_error_analysis(cfg, state)
print("✅ Stage 11 Complete — Error analysis done. Check logs above.")

---
## 🧪 Cell 14 — Stage 12: Generate Test Candidates

In [ ]:
from pipeline import generate_test_candidates

print("⏳ Stage 12: Loading test data & generating test candidates...")
generate_test_candidates(cfg, state)
print(f"✅ Stage 12 Complete — Test S1: {len(state.test_s1_clean)} | Test candidates: {len(state.test_candidates)}")

---
## 🧮 Cell 15 — Stage 13: Build Test Features

In [ ]:
from pipeline import build_test_features

print("⏳ Stage 13: Extracting test features...")
build_test_features(cfg, state)
print(f"✅ Stage 13 Complete — Test features: {state.test_features.shape}")

---
## 🎯 Cell 16 — Stage 14: Run Test Inference

In [ ]:
from pipeline import run_test_inference

print("⏳ Stage 14: Running test inference...")
run_test_inference(cfg, state)
n_with_matches = sum(1 for v in state.test_predictions.values() if v)
print(f"✅ Stage 14 Complete — Predictions: {len(state.test_predictions)} entities | {n_with_matches} with matches")

---
## 📝 Cell 17 — Stage 15: Generate Submission Files

In [ ]:
from pipeline import create_submission_files

print("⏳ Stage 15: Generating submission files...")
create_submission_files(cfg, state)
print("\n✅ Stage 15 Complete — Submission files generated!")
print(f"   📄 {cfg.data.output_dir}/matching_results.tsv")
print(f"   📄 {cfg.data.output_dir}/candidate_pairs.tsv")

---
## ✅ Cell 18: Inspect Submission Files

In [ ]:
from IPython.display import display
import pandas as pd

matching_df = pd.read_csv(f"{OUTPUT_DIR}/matching_results.tsv", sep="\t", dtype=str, keep_default_na=False)
candidate_df = pd.read_csv(f"{OUTPUT_DIR}/candidate_pairs.tsv", sep="\t", dtype=str, keep_default_na=False)

print(f"📊 matching_results.tsv : {len(matching_df)} rows")
print(f"   Matches found         : {(matching_df['matched_entity_ids'] != '').sum()}")
print(f"   Singletons            : {(matching_df['matched_entity_ids'] == '').sum()}")
print("-" * 50)
print(f"📊 candidate_pairs.tsv   : {len(candidate_df)} rows")
print("-" * 50)
print("\nFirst 10 rows of matching_results.tsv:")
display(matching_df.head(10))

---
## 🔄 Cell 19: Switch to Full Dataset (After Sanity Check Passes)

**Run this cell ONLY after the sanity check above completes successfully.**

It resets `SAMPLE_SIZE` to `None` (full dataset) and reinitializes the pipeline state. Then re-run Cells 3–18 for the full pipeline.

In [ ]:
# =============================================
# SWITCH TO FULL DATASET RUN
# =============================================
# After running this cell, re-run Cell 3 through Cell 18

SAMPLE_SIZE = None  # Full dataset
cfg.runtime.debug_sample_size = None
state = PipelineState()  # Fresh state

print("🔄 Config updated for FULL DATASET RUN")
print("   Sample size: FULL DATASET (no limit)")
print("\n👉 Now re-run Cell 3 through Cell 18 for the full pipeline!")